# Wide Rectangles (Showing the List is Exhaustive)

This notebook contains work toward verifying that the list of MFAs in 2_1_cleaned_architectures.csv with $L=7$ is a complete list.

For the write-up see [verify_exhaustive_list.md](verify_exhaustive_list.md).

The code here was verified by humans with code generated by Github Copilot and/or Gemini.

## Find Uncovered Regions

Due to the monotonicity property of being filling, and how we are searching for MFAs, we can easily determine which regions we need to search for potentially missed MFAs.

In [1]:
import math

def get_thm14_floor(d0, dL, L):
    """Use Theorem 14 & Proposition 17 of KTB to find the lower bounds on width needed to be filling"""
    floor_bounds = []
    full_degree = 2**(L - 1)
    ambient_dim = dL * math.comb(d0 + full_degree - 1, full_degree)
    
    for i in range(1, L):
        deg_first = 2**(i - 1)
        deg_second = 2**(L - 1 - i)
        d_i = 1
        while True:
            A1 = d_i * math.comb(d0 + deg_first - 1, deg_first)
            A2 = dL * math.comb(d_i + deg_second - 1, deg_second)
            if A1 + A2 - d_i < ambient_dim:
                d_i += 1
            else:
                floor_bounds.append(d_i)
                break
    return floor_bounds

In [2]:
def get_uncovered_regions(d0, dL, L, known_minimals):
    """
    Determines the infinite rectangles / infinite regions that we need to check
    """
    floor_bounds = get_thm14_floor(d0, dL, L)
    k = L - 1 # Number of hidden layers
    
    # Start with N^{L-1}
    ceilings = [[float('inf')] * k]

    """
    To get the uncovered regions,  we have three loops to do this.
    We are essentially trying to enumerate all possible "ceilings" of these infinite boxes.
    """
    
    for t in known_minimals:
        new_ceilings = []
        for c in ceilings:
            # if the current ceiling strictly dodges the minimal tuple 't' in at least one dimension 
            # if the current ceiling cannot dodge, then it's a subarchitecture and can be ignored
            if any(c[i] <= t[i] - 1 for i in range(k)):
                new_ceilings.append(c)
                continue
            
            for i in range(k):
                # only keep if the new ceiling doesn't drop below the mathematical floor obtained from Thm 14
                if t[i] - 1 >= floor_bounds[i]:
                    new_c = list(c)
                    new_c[i] = t[i] - 1
                    new_ceilings.append(new_c)
                    
        # remove redundant sub-regions
        filtered_ceilings = []
        for c in new_ceilings:
            is_dominated = False
            for other in new_ceilings:
                if c != other and all(other[j] >= c[j] for j in range(k)):
                    is_dominated = True
                    break
            if not is_dominated:
                filtered_ceilings.append(c)
                
        ceilings = filtered_ceilings

    return floor_bounds, ceilings

In [3]:
def format_regions(floor_bounds, ceilings):
    """Prints the intervals so it can be copy-pasted for LaTeX usage effectively"""
    if not ceilings:
        return "Space is completely covered. No unbounded regions remain."
        
    output = []
    for idx, c in enumerate(ceilings):
        region = []
        for i in range(len(floor_bounds)):
            lower = floor_bounds[i]
            
            # Format infinity as 'inf', and ensure finite bounds are integers
            upper = "\\infty" if c[i] == float('inf') else str(int(c[i]))
            
            # Always format as an interval [min, max]
            if upper == "\\infty":
                region.append(f"[{lower}, {upper})")
            else:
                region.append(f"[{lower}, {upper}]")
                
                
        # Join with the LaTeX \times string
        # Had to format a couple of times to get a version readable for LaTeX
        formatted_region = r" \times ".join(region)
        output.append(f"\\text{{Region {idx + 1}:}}\\qquad & \t{formatted_region}\\\\")
        
    return "\n".join(output)

In [4]:

d0 = 2
dL = 1
L = 7 # L=6 means 5 hidden layers

# Example list of known minimal filling architectures
# Format: Tuples of hidden layer widths (excluding d0 and dL)
known_minimals = [(3, 4, 5, 4, 6, 4),(3, 4, 5, 6, 4, 2), (3, 3, 4, 5, 6, 3), (3, 4, 5, 5, 5, 2),
      (3, 3, 5, 6, 4, 4),(3, 3, 5, 7, 4, 2), (3, 3, 4, 6, 5, 2),(3, 4, 5, 5, 4, 4),
      (3, 4, 4, 5, 5, 4),(3, 3, 6, 6, 4, 3),(3, 3, 5, 5, 5, 4),(3, 4, 6, 5, 4, 3),
      (3, 5, 5, 5, 4, 3)
]

floor_bounds, uncovered_ceilings = get_uncovered_regions(d0, dL, L, known_minimals)

print(f"Network Structure: d0={d0}, L={L}, dL={dL}")
print(f"Theorem 14 Floor:  {floor_bounds}\n")
print("Exact Uncovered Mathematical Regions:")
print("-" * 50)
print(format_regions(floor_bounds, uncovered_ceilings))

Network Structure: d0=2, L=7, dL=1
Theorem 14 Floor:  [3, 3, 4, 4, 4, 2]

Exact Uncovered Mathematical Regions:
--------------------------------------------------
\text{Region 1:}\qquad & 	[3, \infty) \times [3, 3] \times [4, \infty) \times [4, 4] \times [4, \infty) \times [2, \infty)\\
\text{Region 2:}\qquad & 	[3, \infty) \times [3, 3] \times [4, 4] \times [4, 5] \times [4, 5] \times [2, \infty)\\
\text{Region 3:}\qquad & 	[3, \infty) \times [3, 3] \times [4, \infty) \times [4, 5] \times [4, 4] \times [2, \infty)\\
\text{Region 4:}\qquad & 	[3, \infty) \times [3, 3] \times [4, \infty) \times [4, 5] \times [4, 5] \times [2, 3]\\
\text{Region 5:}\qquad & 	[3, \infty) \times [3, 3] \times [4, 5] \times [4, 6] \times [4, 4] \times [2, 3]\\
\text{Region 6:}\qquad & 	[3, \infty) \times [3, 3] \times [4, \infty) \times [4, 6] \times [4, 4] \times [2, 2]\\
\text{Region 7:}\qquad & 	[3, \infty) \times [3, 3] \times [4, \infty) \times [4, 5] \times [4, \infty) \times [2, 2]\\
\text{Region 8:}\

## Quick Proof of Nonfillingness of Certain Regions

This code below was generated using Gemini. It's a very simple for loop which just computes all of the possible inequalities using the recursive bound. We verified the code and verified the inequaltiies produced.
We were mainly interested in producing bounds for each of the possibilities and we only needed to test each ``cut-off" for the recursive bound.

In [5]:
import math
import itertools

In [6]:
def compute_ambient(din, dout, depth):
    """Computes the ambient dimension of an architecture."""
    deg = 2**(depth - 1)
    return dout * math.comb(din + deg - 1, deg)


In [7]:
def evaluate_cut_subset(d0, dL, L, cut_subset):
    """
    Uses the Recursive Bound
    cut_subset: list of tuples (layer_index, max_width)
    """
    cuts = [(0, d0)] + list(cut_subset) + [(L, dL)]
    
    sum_A = 0
    sum_cuts = 0
    
    for m in range(len(cuts) - 1):
        idx_in, width_in = cuts[m]
        idx_out, width_out = cuts[m+1]
        depth = idx_out - idx_in
        A = compute_ambient(width_in, width_out, depth)
        sum_A += A
        
    for m in range(1, len(cuts) - 1):
        sum_cuts += cuts[m][1]
        
    return sum_A - sum_cuts, sum_A, sum_cuts

In [8]:
def prove_regions_non_filling(d0, dL, L, ceilings):
    """
    Attempts to rigorously prove that all given regions are non-filling
    by slicing them at finite upper bounds. 
    Outputs a clean, multi-column LaTeX align* format for perfect vertical stacking.
    """
    target_ambient = compute_ambient(d0, dL, L)
    print(f"% Target Total Ambient Dimension: {target_ambient}")
    print("% " + "=" * 60)
    
    all_proven = True
    
    for idx, c in enumerate(ceilings):
        possible_cuts = []
        for i, val in enumerate(c):
            if val != float('inf'):
                possible_cuts.append((i + 1, int(val)))
        
        proven = False
        best_bound = float('inf')
        best_subset = None
        
        # Test all combinations of cuts, starting from the maximum number of cuts
        for r in range(len(possible_cuts), 0, -1):
            for subset in itertools.combinations(possible_cuts, r):
                bound, A_sum, cut_sum = evaluate_cut_subset(d0, dL, L, subset)
                
                if bound < best_bound:
                    best_bound = bound
                    best_subset = subset
                    
                if bound < target_ambient:
                    proven = True
                    break 
            if proven:
                break 
                
        if not proven:
            all_proven = False
            print(f"\\text{{Region {idx + 1}:}} & \\quad \\text{{Unproven}} && \\text{{(Lowest bound: }} {best_bound} \\geq {target_ambient}\\text{{)}} \\\\")
            continue
            
        # 1. Build the full architecture string D_{(...)}
        cut_dict = dict(best_subset)
        full_elements = [str(d0)]
        for j in range(1, L):
            if j in cut_dict:
                full_elements.append(str(cut_dict[j]))
            else:
                full_elements.append(f"d_{{{j}}}")
        full_elements.append(str(dL))
        full_arch_str = ",".join(full_elements)
        d_full_latex = f"D_{{({full_arch_str})}}"
        
        # 2. Build the sub-network strings and compute ambients
        cuts_full = [(0, d0)] + list(best_subset) + [(L, dL)]
        D_parts = []
        A_vals = []
        
        for m in range(len(cuts_full) - 1):
            i_in, w_in = cuts_full[m]
            i_out, w_out = cuts_full[m+1]
            depth = i_out - i_in
            A = compute_ambient(w_in, w_out, depth)
            A_vals.append(A)
            
            part_elements = [str(w_in)]
            for j in range(i_in + 1, i_out):
                part_elements.append(f"d_{{{j}}}")
            part_elements.append(str(w_out))
            part_str = ",".join(part_elements)
            D_parts.append(f"D_{{({part_str})}}")
            
        # 3. Format the bottleneck subtractions
        cut_sum = sum(cut[1] for cut in best_subset)
        cut_str = f" - {cut_sum}" if cut_sum > 0 else ""
            
        d_parts_latex = " + ".join(D_parts)
        a_vals_latex = " + ".join(map(str, A_vals))
        
        # LaTeX formatted multi-column block for align*
        # Mainly for displaying purposes in LaTeX
        print(f"\\text{{Region {idx + 1}:}} & \\quad {d_full_latex} && \\leq {d_parts_latex}{cut_str} \\\\")
        print(f"& && \\leq {a_vals_latex}{cut_str} \\\\")
        print(f"& && = {best_bound} < {target_ambient} \\\\")

    print("% " + "=" * 60)
    if all_proven:
        print("% Every single unbounded region has been mathematically proven to be strictly non-filling.")
    else:
        print("% Some regions could not be bounded below the target ambient. You must find more minimal architectures inside those specific regions.")

In [9]:
d0 = 2
dL = 1
L = 7 

known_minimals = [
    (3, 4, 5, 4, 6, 4), (3, 4, 5, 6, 4, 2), (3, 3, 4, 5, 6, 3), 
    (3, 4, 5, 5, 5, 2), (3, 3, 5, 6, 4, 4), (3, 3, 5, 7, 4, 2), 
    (3, 3, 4, 6, 5, 2), (3, 4, 5, 5, 4, 4), (3, 4, 4, 5, 5, 4), 
    (3, 3, 6, 6, 4, 3), (3, 3, 5, 5, 5, 4), (3, 4, 6, 5, 4, 3), 
    (3, 5, 5, 5, 4, 3)
]

print(f"Network Structure: d0={d0}, L={L}, dL={dL}")
print("-" * 60)

# 1. Generate the exact missing regions
floor_bounds, uncovered_ceilings = get_uncovered_regions(d0, dL, L, known_minimals)
print(f"Theorem 14 Floor: {floor_bounds}\n")
print("Exact Uncovered Mathematical Regions:")
print(format_regions(floor_bounds, uncovered_ceilings))



# 2. Feed those regions directly into the prover
prove_regions_non_filling(d0, dL, L, uncovered_ceilings)

Network Structure: d0=2, L=7, dL=1
------------------------------------------------------------
Theorem 14 Floor: [3, 3, 4, 4, 4, 2]

Exact Uncovered Mathematical Regions:
\text{Region 1:}\qquad & 	[3, \infty) \times [3, 3] \times [4, \infty) \times [4, 4] \times [4, \infty) \times [2, \infty)\\
\text{Region 2:}\qquad & 	[3, \infty) \times [3, 3] \times [4, 4] \times [4, 5] \times [4, 5] \times [2, \infty)\\
\text{Region 3:}\qquad & 	[3, \infty) \times [3, 3] \times [4, \infty) \times [4, 5] \times [4, 4] \times [2, \infty)\\
\text{Region 4:}\qquad & 	[3, \infty) \times [3, 3] \times [4, \infty) \times [4, 5] \times [4, 5] \times [2, 3]\\
\text{Region 5:}\qquad & 	[3, \infty) \times [3, 3] \times [4, 5] \times [4, 6] \times [4, 4] \times [2, 3]\\
\text{Region 6:}\qquad & 	[3, \infty) \times [3, 3] \times [4, \infty) \times [4, 6] \times [4, 4] \times [2, 2]\\
\text{Region 7:}\qquad & 	[3, \infty) \times [3, 3] \times [4, \infty) \times [4, 5] \times [4, \infty) \times [2, 2]\\
\text{Re